# Experiment 1: beam_search_small

**Hypothesis:** Beam search and better decoding parameters reduce hallucinations and repetition compared to greedy decoding.

**Model:** `openai/whisper-small` (244M params) — same as baseline

**Change:** Only decoding parameters (num_beams=5, no_repeat_ngram_size=3, condition_on_prev_tokens=False)

**Output:** `submissions/submission_beam_small.csv`

## 1. Install Dependencies

In [3]:
!pip install -q soundfile "datasets==3.2.0" transformers
print("Dependencies installed.")

Dependencies installed.


In [4]:
import os, csv, time
from pathlib import Path

import datasets
import numpy as np
import torch
import transformers
from tqdm.auto import tqdm

PROJECT_ROOT = Path(".").resolve().parent
DATA_DIR = PROJECT_ROOT / "data"

env_file = PROJECT_ROOT / ".env"
if env_file.exists():
    for line in env_file.read_text().strip().splitlines():
        if "=" in line and not line.startswith("#"):
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip()
    print("HF_TOKEN loaded from .env")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

HF_TOKEN loaded from .env
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB


## 3. Load Test Data

In [6]:
LANGUAGES = ["lug", "lin", "sna"]
SAMPLE_RATE = 16_000

WHISPER_LANG_MAP = {
    "lug": "swahili",
    "lin": "lingala",
    "sna": "shona",
}

print("Loading test data from local parquet files...\n")
test_data = {}
for lang in LANGUAGES:
    shards = sorted(DATA_DIR.glob(f"{lang}-test-*.parquet"))
    if not shards:
        print(f"  WARNING: No parquet files for {lang}")
        continue
    print(f"  {lang}: {len(shards)} shard(s) ...", end=" ", flush=True)
    shard_ds = [datasets.Dataset.from_parquet(str(s)) for s in shards]
    ds = datasets.concatenate_datasets(shard_ds) if len(shard_ds) > 1 else shard_ds[0]
    ds = ds.cast_column("audio", datasets.Audio(sampling_rate=SAMPLE_RATE))
    test_data[lang] = ds
    print(f"OK ({len(ds)} examples)")
print("\nDone.")

Loading test data from local parquet files...

  lug: 1 shard(s) ... OK (638 examples)
  lin: 2 shard(s) ... OK (1866 examples)
  sna: 2 shard(s) ... OK (1749 examples)

Done.


## 4. Load Whisper Small

In [7]:
MODEL_ID = "openai/whisper-small"

processor = transformers.WhisperProcessor.from_pretrained(MODEL_ID)
model = transformers.WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16
)

model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

model = model.to(device)
model.eval()

params = sum(p.numel() for p in model.parameters()) / 1e6
gpu_mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
print(f"Model: {MODEL_ID} ({params:.1f}M params)")
print(f"GPU memory: {gpu_mem:.2f} GB")

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 816.45it/s] 


Model: openai/whisper-small (241.7M params)
GPU memory: 0.50 GB


## 5. Generate Submission — Beam Search Decoding

**Changed vs baseline:** `num_beams=5`, `no_repeat_ngram_size=3`, `condition_on_prev_tokens=False`, `length_penalty=1.0`

In [ ]:
test_csv_path = PROJECT_ROOT / "Test.csv"
sample_csv_path = PROJECT_ROOT / "SampleSubmission.csv"
submission_dir = PROJECT_ROOT / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)
submission_path = submission_dir / "submission_beam_small.csv"

# Read test IDs
test_ids = []
with open(test_csv_path, "r", encoding="utf-8") as fh:
    for row in csv.DictReader(fh):
        test_ids.append(row["ID"])

lang_to_ids = {}
for tid in test_ids:
    lang = tid.split("_")[0]
    lang_to_ids.setdefault(lang, []).append(tid)

print(f"Test set: {len(test_ids)} samples")
for lang, ids in lang_to_ids.items():
    print(f"  {lang}: {len(ids)} samples")

# Inference
predictions = {}
start_time = time.time()

for lang in LANGUAGES:
    if lang not in lang_to_ids or lang not in test_data:
        continue

    ds = test_data[lang]
    needed_ids = set(lang_to_ids[lang])
    whisper_lang = WHISPER_LANG_MAP[lang]

    id_lookup = {}
    for idx in range(len(ds)):
        raw_id = str(ds[idx]["id"])
        full_id = f"{lang}_{raw_id}" if not raw_id.startswith(lang) else raw_id
        if full_id in needed_ids:
            id_lookup[idx] = full_id

    print(f"\n{lang}: transcribing {len(id_lookup)} samples (lang={whisper_lang})...")

    for idx in tqdm(sorted(id_lookup.keys()), desc=f"Predict {lang}"):
        example = ds[idx]
        audio_array = np.asarray(example["audio"]["array"], dtype=np.float32)

        input_features = processor.feature_extractor(
            audio_array, sampling_rate=SAMPLE_RATE, return_tensors="pt",
        ).input_features.to(device=device, dtype=torch.float16)

        with torch.no_grad():
            pred_ids = model.generate(
                input_features,
                max_new_tokens=225,
                language=whisper_lang,
                task="transcribe",
                num_beams=5,
                no_repeat_ngram_size=3,
                condition_on_prev_tokens=False,
                length_penalty=1.0,
            )

        transcript = processor.tokenizer.decode(
            pred_ids[0], skip_special_tokens=True
        ).strip()
        predictions[id_lookup[idx]] = transcript

elapsed = time.time() - start_time
gpu_mem_peak = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0

# Write submission
with open(submission_path, "w", encoding="utf-8", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["ID", "Target"])
    for tid in test_ids:
        writer.writerow([tid, predictions.get(tid, "")])

print(f"\nSubmission written to: {submission_path}")
print(f"Predictions: {len(predictions)} / {len(test_ids)}")
print(f"Runtime: {elapsed:.0f}s ({elapsed/60:.1f}min)")
print(f"Peak GPU memory: {gpu_mem_peak:.2f} GB")

# Quick stats
non_empty = [v for v in predictions.values() if v.strip()]
avg_len = sum(len(t) for t in non_empty) / max(len(non_empty), 1)
print(f"Avg transcript length: {avg_len:.0f} chars")

# Validate
if sample_csv_path.exists():
    with open(sample_csv_path, "r", encoding="utf-8") as fh:
        expected = {row["ID"] for row in csv.DictReader(fh)}
    with open(submission_path, "r", encoding="utf-8") as fh:
        submitted = {row["ID"] for row in csv.DictReader(fh)}
    missing = expected - submitted
    empty = sum(1 for tid in test_ids if not predictions.get(tid, "").strip())
    if missing:
        print(f"VALIDATION FAILED: Missing {len(missing)} IDs!")
    elif empty:
        print(f"WARNING: {empty} IDs have empty transcriptions")
    else:
        print("VALIDATION PASSED")

Test set: 4253 samples
  lug: 638 samples
  lin: 1866 samples
  sna: 1749 samples

lug: transcribing 638 samples (lang=swahili)...


Predict lug:   0%|          | 0/638 [00:00<?, ?it/s][transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created i

## 6. Quality Report

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.quality_analysis import analyze

report = analyze(str(submission_path), "beam_search_small")

report_path = PROJECT_ROOT / "reports" / "quality_report_beam_search_small.md"
report_path.parent.mkdir(exist_ok=True)
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print(report)